In [1]:
# Install dependencies (run in cell 1)
!pip install google-drive-cli anthropic unsloth -q

# Mount Google Drive (run in cell 2)
from google.colab import drive
drive.mount('/content/drive')

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.4/75.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 70.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.4/22.4 MB 89.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 122.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 92.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 125.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 123.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━

In [14]:
!pip install anthropic langchain langchain-community requests unsloth -q

In [20]:
# Patch vector_store.py to avoid chromadb import
patch = '''
def query_filing(ticker: str, query: str, n_results: int = 5):
    """Stub — not needed in judge-only mode."""
    return []
'''

with open("/content/fin-research-agent/retrieval/vector_store.py", "w") as f:
    f.write(patch)

print("Patched vector_store.py")

Patched vector_store.py


In [13]:
!pip install chromadb langchain-openai langgraph sentence-transformers -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 67.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 121.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 85.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [7]:
!ls -la /content/drive/MyDrive/fin-research-agent/agents/

ls: cannot access '/content/drive/MyDrive/fin-research-agent/agents/': No such file or directory


In [18]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/fin-research-agent/fin-research-agent-full.zip"
extract_path = "/content"

print(f"Extracting {zip_path}...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Done. Checking extracted structure:")
!ls -la /content/fin-research-agent/agents/ | head -20

Extracting /content/drive/MyDrive/fin-research-agent/fin-research-agent-full.zip...
Done. Checking extracted structure:
total 48
drwxr-xr-x  4 root root 4096 Sep 13 12:02 .
drwxr-xr-x 15 root root 4096 Sep 13 12:02 ..
-rw-r--r--  1 root root 2579 Sep 13 12:07 analyst_agent.py
-rw-r--r--  1 root root 2412 Sep 13 12:07 data_agent.py
-rw-r--r--  1 root root 2794 Sep 13 12:07 graph.py
-rw-r--r--  1 root root 5511 Sep 13 12:07 judge_agent.py
-rw-r--r--  1 root root  826 Sep 13 12:07 llm_client.py
drwxr-xr-x  3 root root 4096 Sep 13 12:02 prompts
drwxr-xr-x  2 root root 4096 Sep 13 12:05 __pycache__
-rw-r--r--  1 root root 1417 Sep 13 12:07 report_agent.py
-rw-r--r--  1 root root 3045 Sep 13 12:07 retriever_agent.py


In [17]:
!pip install opentelemetry-sdk==1.27.0 opentelemetry-api==1.27.0 opentelemetry-exporter-otlp-proto-grpc==1.27.0 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.0/64.0 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.5/52.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 28.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
wandb 0.28.1 requires protobuf!=5.28.0,!=5.29.0,<8,>=5, but you have protobuf 4.25.9 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.9 which is incompatible.
grain 0.2.18 requires protobuf>=5.28.3, but you have protobuf 4.25.9 which is incompatible.
tensorflow 2.20.0 requires protobuf>=5.28.0, but you have protobuf 4.25.9 which is incompatible.
google-adk 2.7.1 requ

In [22]:
import json
import os
import sys
import time
from scipy.stats import pearsonr

# Paths
PROJECT_PATH = "/content/fin-research-agent"
ANALYST_PATH = "/content/drive/MyDrive/fin-research-agent/analyst_outputs.json"
RESULTS_PATH = f"{PROJECT_PATH}/results/judge_agreement.json"

os.chdir(PROJECT_PATH)
sys.path.insert(0, PROJECT_PATH)

# Load analyst outputs
with open(ANALYST_PATH) as f:
    analyst_data = json.load(f)

print(f"Loaded {len(analyst_data['outputs'])} analyst outputs\n")

# Initialize results
if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH) as f:
        results = json.load(f)
else:
    results = {"entries": [], "aggregate_stats": {}}

processed = {e["ticker"] for e in results["entries"] if e.get("api_scores") and e.get("finetuned_scores")}
print(f"Already processed: {processed}\n")

# Import judge_agent_node directly — skip the full graph
from agents.judge_agent import judge_agent_node

# Process each ticker
for entry in analyst_data['outputs']:
    ticker = entry["ticker"]
    report_draft = entry["report_draft"]

    if not report_draft:
        print(f"✗ {ticker} has no report_draft — skipping")
        continue

    if ticker in processed:
        print(f"✓ {ticker} already scored — skipping")
        continue

    print(f"\n{'='*60}")
    print(f"Processing {ticker}")
    print(f"{'='*60}")

    state = {
        "ticker": ticker,
        "question": f"What are the main risks and growth opportunities for {ticker}?",
        "report_draft": report_draft,
        "retrieved_chunks": []  # empty — judge uses report_draft, not chunks
    }

    # Run API judge
    print(f"  Running API judge...")
    os.environ["JUDGE_MODE"] = "api"
    try:
        api_result = judge_agent_node(state)
        api_scores = api_result.get("judge_score", {})
        print(f"    ✓ Overall: {api_scores.get('overall')}, Flagged: {len(api_scores.get('flagged_issues', []))}")
    except Exception as e:
        print(f"    ✗ Failed: {e}")
        api_scores = {}

    time.sleep(5)

    # Run finetuned judge
    print(f"  Running finetuned judge...")
    os.environ["JUDGE_MODE"] = "finetuned"
    try:
        ft_result = judge_agent_node(state)
        ft_scores = ft_result.get("judge_score", {})
        print(f"    ✓ Overall: {ft_scores.get('overall')}, Flagged: {len(ft_scores.get('flagged_issues', []))}")
    except Exception as e:
        print(f"    ✗ Failed: {e}")
        ft_scores = {}

    # Save if both succeeded
    if api_scores and ft_scores:
        score_diffs = {}
        for metric in ["grounding", "completeness", "clarity", "overall"]:
            api_val = api_scores.get(metric)
            ft_val = ft_scores.get(metric)
            if api_val is not None and ft_val is not None:
                score_diffs[metric] = api_val - ft_val

        both_flagged = len(api_scores.get("flagged_issues", [])) > 0 and len(ft_scores.get("flagged_issues", [])) > 0

        results["entries"].append({
            "ticker": ticker,
            "api_scores": api_scores,
            "finetuned_scores": ft_scores,
            "score_diffs": score_diffs,
            "both_flagged": both_flagged
        })

        with open(RESULTS_PATH, "w") as f:
            json.dump(results, f, indent=2)

        print(f"  ✓ Saved {ticker}")
    else:
        print(f"  ✗ Skipping {ticker} — one or both judges failed")

    time.sleep(10)

# Compute aggregate stats
print(f"\n{'='*60}\nComputing aggregate stats...\n{'='*60}\n")

api_by_metric = {m: [] for m in ["grounding", "completeness", "clarity", "overall"]}
ft_by_metric  = {m: [] for m in ["grounding", "completeness", "clarity", "overall"]}
diffs_by_metric = {m: [] for m in ["grounding", "completeness", "clarity", "overall"]}
both_flagged_count = 0

for entry in results["entries"]:
    for metric in ["grounding", "completeness", "clarity", "overall"]:
        a = entry["api_scores"].get(metric)
        f = entry["finetuned_scores"].get(metric)
        if a is not None and f is not None:
            api_by_metric[metric].append(a)
            ft_by_metric[metric].append(f)
            diffs_by_metric[metric].append(abs(a - f))
    if entry.get("both_flagged"):
        both_flagged_count += 1

pearson_corr = {}
for metric in ["grounding", "completeness", "clarity", "overall"]:
    if len(api_by_metric[metric]) > 1:
        try:
            corr, pval = pearsonr(api_by_metric[metric], ft_by_metric[metric])
            pearson_corr[metric] = {"correlation": round(corr, 3), "p_value": round(pval, 3)}
        except:
            pearson_corr[metric] = None

pct_within = {1: {}, 2: {}}
for metric in ["grounding", "completeness", "clarity", "overall"]:
    for threshold in [1, 2]:
        if diffs_by_metric[metric]:
            within = sum(1 for d in diffs_by_metric[metric] if d <= threshold)
            pct_within[threshold][metric] = round(100 * within / len(diffs_by_metric[metric]), 1)

pct_both_flagged = round(100 * both_flagged_count / len(results["entries"]), 1) if results["entries"] else 0

results["aggregate_stats"] = {
    "tickers_scored": len(results["entries"]),
    "pearson_correlation": pearson_corr,
    "pct_within_1_point": pct_within[1],
    "pct_within_2_points": pct_within[2],
    "pct_both_flagged_issues": pct_both_flagged
}

with open(RESULTS_PATH, "w") as f:
    json.dump(results, f, indent=2)

print(f"✓ Results saved to {RESULTS_PATH}")
print(f"\nAggregate Stats:\n{json.dumps(results['aggregate_stats'], indent=2)}")

Loaded 12 analyst outputs

Already processed: set()


Processing AAPL
  Running API judge...
    ✓ Overall: 4, Flagged: 7
  Running finetuned judge...
    ✓ Overall: 4, Flagged: 6
  ✓ Saved AAPL

Processing TSLA
  Running API judge...
    ✓ Overall: 2, Flagged: 4
  Running finetuned judge...
    ✓ Overall: 4, Flagged: 4
  ✓ Saved TSLA

Processing NFLX
  Running API judge...
    ✓ Overall: 4, Flagged: 7
  Running finetuned judge...
    ✓ Overall: 4, Flagged: 7
  ✓ Saved NFLX

Processing GOOGL
  Running API judge...
    ✓ Overall: 3, Flagged: 3
  Running finetuned judge...
    ✓ Overall: 0, Flagged: 3
  ✓ Saved GOOGL

Processing META
  Running API judge...
    ✓ Overall: 4, Flagged: 7
  Running finetuned judge...
    ✓ Overall: 2, Flagged: 8
  ✓ Saved META

Processing AMZN
  Running API judge...
    ✓ Overall: 0, Flagged: 4
  Running finetuned judge...
    ✓ Overall: 1, Flagged: 3
  ✓ Saved AMZN

Processing MSFT
  Running API judge...
    ✓ Overall: 4, Flagged: 5
  Running finetuned judg

In [23]:
from google.colab import files
files.download("/content/fin-research-agent/results/judge_agreement.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
!ls -la /content/drive/MyDrive/fin-research-agent/
!find /content/drive/MyDrive/fin-research-agent -name "analyst_outputs.json" -type f

total 16
drwx------ 4 root root 4096 Sep  1 05:04 .
drwx------ 4 root root 4096 Sep 13 11:37 ..
drwx------ 2 root root 4096 Sep  1 09:46 adapters
drwx------ 2 root root 4096 Sep  1 05:04 finetune


In [4]:
!find /content/drive/MyDrive/fin-research-agent -name "analyst_outputs.json" -type f

/content/drive/MyDrive/fin-research-agent/analyst_outputs.json


In [8]:
!ls -la /content/drive/MyDrive/fin-research-agent/

total 27
drwx------ 4 root root  4096 Sep  1 05:04 .
drwx------ 4 root root  4096 Sep 13 11:37 ..
drwx------ 3 root root  4096 Sep  1 09:46 adapters
-rw------- 1 root root 10389 Sep 13 07:50 analyst_outputs.json
drwx------ 3 root root  4096 Sep  1 05:04 finetune


In [10]:
!find /content/drive/MyDrive -name "fin-research-agent-full.zip" -type f

/content/drive/MyDrive/fin-research-agent/fin-research-agent-full.zip
